# Production Method Comparison (Economic Analysis)

Live economic comparison of vanilla and modded production methods, grouped by slots.

In [ ]:
import pandas as pd
import numpy as np

from core.parser.path_resolver import PathResolver
from core.data.building_data import BuildingData
from core.data.goods_data import GoodsData
from core.data.defines_data import DefinesData
from core.data.pop_data import PopData
from analysis.building_levels.building_analysis import load_config

config = load_config()
path_resolver = PathResolver(config['game_path'], config['mod_path'])
goods_data = GoodsData(path_resolver)
building_data = BuildingData(path_resolver)
defines_data = DefinesData(path_resolver)
pop_data = PopData(path_resolver)

goods_data.load_all()
building_data.load_all()
defines_data.load_all()
pop_data.load_all()

pd.options.display.max_columns = None
pd.options.display.precision = 2
pd.options.display.float_format = '{:.2f}'.format

In [ ]:
# Resolved food good (good with food > 0) and its price; fallback to defines
food_good_vanilla = goods_data.get_food_good(False)
food_good_modded = goods_data.get_food_good(True)
food_price_vanilla = goods_data.get_food_good_price(False)
food_price_modded = goods_data.get_food_good_price(True)
if food_price_vanilla is None:
    food_price_vanilla = defines_data.get_vanilla_define("NMarket", "FOOD_PRICE")
if food_price_modded is None:
    food_price_modded = defines_data.get_define("NMarket", "FOOD_PRICE")
print(f"Vanilla food good: {food_good_vanilla}, price: {food_price_vanilla}")
print(f"Modded food good:  {food_good_modded}, price: {food_price_modded}")

In [ ]:
pop_comparison = pd.merge(
    pop_data.vanilla_df[['pop_food_consumption']].rename(columns={'pop_food_consumption': 'Vanilla_Food'}),
    pop_data.modded_df[['pop_food_consumption']].rename(columns={'pop_food_consumption': 'Modded_Food'}),
    left_index=True,
    right_index=True
)
pop_comparison['Change'] = (pop_comparison['Modded_Food'] / pop_comparison['Vanilla_Food']) - 1
pop_comparison.style.format({'Change': '{:+.1%}'}).background_gradient(subset=['Change'], cmap='RdYlGn_r')

In [ ]:
# goods_data.vanilla_df.columns
# print(goods_data.vanilla_df.shape)
goods_data.vanilla_df[["method", "default_market_price", "transport_cost", "food"]].sort_values(by="food", ascending=False).head(15)
# goods_data.vanilla_df[["method", "default_market_price", "transport_cost", "food"]].sort_values(by="food", ascending=False).head(40)


# for element in goods_data.vanilla_df.reset_index().name.tolist():
#     print(element)

In [ ]:
# goods_data.modded_df.columns
# print(goods_data.modded_df.shape)
# print(goods_data.modded_df.columns)
goods_data.modded_df[["method", "default_market_price", "transport_cost", "food"]].sort_values(by="food", ascending=False).tail(15)
# goods_data.modded_df[["method", "default_market_price", "transport_cost", "food"]].sort_values(by="transport_cost", ascending=True).head(40)

In [ ]:
def get_slot_df(building_name, slot_index, is_modded=True):
    comp = building_data.compare_production_methods(building_name, goods_data=goods_data, defines_data=defines_data, pop_data=pop_data)
    slots = comp['modded_slots'] if is_modded else comp['vanilla_slots']
    
    if slot_index >= len(slots):
        return pd.DataFrame()
    
    # Get building definition for employment info
    b_def = building_data.get_building(building_name) if is_modded else building_data.get_vanilla_building(building_name)
    b_def = b_def or {}
    
    slot = slots[slot_index]
    rows = []
    # System columns that shouldn't be prefixed with In_
    system_cols = [
        'profit', 'profit_margin', 'input_cost', 'output_value', 
        'worker_food_cost', 'is_modifier_output', 'output_price', 'modifier_food_output'
    ]
    
    for pm_name, pm in slot.items():
        row = {
            "Version": "Modded" if is_modded else "Vanilla",
            "PM": pm_name,
            "Pop_Type": b_def.get('pop_type', 'unknown'),
            "Employment (1k)": b_def.get('employment_size_val', 0),
            "EPE": pm.get('epe', 0),
            "Profit": pm.get('profit', 0),
            "Margin": pm.get('profit_margin', 0),
            "Input_Cost": pm.get('input_cost', 0),
            "Output_Val": pm.get('output_value', 0),
            "Worker_Food_Cost": pm.get('worker_food_cost', 0)
        }
        for key, val in pm.items():
            if key in ['produced', 'output', 'category']:
                continue
            
            if key in system_cols:
                row[key] = val
            elif not key.startswith(('profit', 'input_cost', 'output_value', 'worker_food_cost', 'output_price', 'is_modifier_output', 'epe')):
                # Prefix goods with In_
                row[f"In_{key}"] = val
                
        # Handle Output columns
        prod = pm.get('produced')
        if prod and not pd.isna(prod):
            row[f"Out_{prod}"] = pm.get('output', 0)
        modifier_food = pm.get('modifier_food_output', 0)
        if modifier_food and modifier_food != 0:
            row["Out_food"] = modifier_food
            
        rows.append(row)
    return pd.DataFrame(rows)

def analyze_building(building_name):
    comp = building_data.compare_production_methods(building_name, goods_data=goods_data, defines_data=defines_data, pop_data=pop_data)
    num_slots = max(len(comp['vanilla_slots']), len(comp['modded_slots']))
    
    slot_dfs = []
    for i in range(num_slots):
        v_df = get_slot_df(building_name, i, is_modded=False)
        m_df = get_slot_df(building_name, i, is_modded=True)
        combined = pd.concat([v_df, m_df], ignore_index=True)
        if combined.empty: continue
        
        # Cleanup zero columns
        for col in combined.columns:
            if col.startswith(('In_', 'Out_')):
                if (combined[col].fillna(0).abs() < 1e-6).all():
                    combined = combined.drop(columns=[col])
        
        meta = ["Version", "PM", "Pop_Type", "Employment (1k)", "EPE"]
        econ = ["Profit", "Margin", "Input_Cost", "Output_Val", "Worker_Food_Cost", "output_price"]
        goods = sorted([c for c in combined.columns if c not in meta + econ])
        
        final_df = combined[meta + econ + goods].fillna(0)
        
        # Explicitly format all numeric columns to 3 decimal places
        float_cols = final_df.select_dtypes(include=[np.number]).columns.tolist()
        format_dict = {col: "{:.3f}" for col in float_cols}
        if 'Margin' in format_dict: 
            format_dict['Margin'] = "{:+.3%}"
        if 'EPE' in format_dict:
            format_dict['EPE'] = "{:+.3%}"
            
        styled = final_df.style.format(format_dict)
        
        # Add gradients for Margin, Profit and EPE
        if 'Margin' in final_df.columns:
            styled = styled.background_gradient(subset=['Margin'], cmap='RdYlGn', vmin=-0.5, vmax=0.5)
        if 'EPE' in final_df.columns:
            # For EPE, positive means we need more efficiency (bad current state), negative means we are already above break-even
            # So we invert the colormap: negative EPE is Green, positive is Red
            styled = styled.background_gradient(subset=['EPE'], cmap='RdYlGn_r', vmin=-0.5, vmax=0.5)
        if 'Profit' in final_df.columns:
            # For profit, we use a divergent map centered around 0
            # We'll use the max absolute profit to center the scale
            max_prof = final_df['Profit'].abs().max()
            styled = styled.background_gradient(subset=['Profit'], cmap='RdYlGn', vmin=-max_prof, vmax=max_prof)
        
        slot_dfs.append(styled)
    return slot_dfs

## Farming Village

In [ ]:
farming_village_analysis = analyze_building("farming_village")
for df in farming_village_analysis: display(df)

## Fishing Village

In [ ]:
fishing_village_analysis = analyze_building("fishing_village")
for df in fishing_village_analysis: display(df)

## Forest Village

In [ ]:
forest_village_analysis = analyze_building("forest_village")
for df in forest_village_analysis: display(df)

## Fruit Orchard

In [ ]:
fruit_orchard_analysis = analyze_building("fruit_orchard")
for df in fruit_orchard_analysis: display(df)

## Sheep Farm

In [ ]:
sheep_farms_analysis = analyze_building("sheep_farms")
for df in sheep_farms_analysis: display(df)

## Cookery

In [ ]:
cookery_analysis = analyze_building("cookery")
for df in cookery_analysis: display(df)

In [ ]:
# Food contribution by input cost: victuals -> 60 food (tavern)
# Attribute % of finished food to each input good by its share of input cost
VICTUALS_TO_FOOD = 60

comp = building_data.compare_production_methods(
    "cookery", goods_data=goods_data, defines_data=defines_data, pop_data=pop_data
)
recipe_slot = comp["modded_slots"][0]

system_cols = [
    "profit", "profit_margin", "input_cost", "output_value",
    "worker_food_cost", "is_modifier_output", "output_price", "modifier_food_output",
    "epe", "produced", "output", "category",
]

rows_first = []  # PM | Good | Amount | Cost | Pct_of_food

for pm_name, pm in recipe_slot.items():
    victuals = pm.get("output", 0.0) if pm.get("produced") == "victuals" else 0.0
    victuals = float(victuals) if victuals else 0.0
    food_out = victuals * VICTUALS_TO_FOOD

    good_costs = []
    for key, val in pm.items():
        if key in system_cols or key.startswith(("profit", "input_cost", "output_value", "worker_food_cost", "output_price", "is_modifier_output", "epe")):
            continue
        good = key
        amount = float(val) if isinstance(val, (int, float)) and not pd.isna(val) else 0.0
        if amount <= 0:
            continue
        price = 0.0
        if good in goods_data.modded_df.index:
            price = float(goods_data.modded_df.loc[good, "default_market_price"]) if "default_market_price" in goods_data.modded_df.columns else 0.0
        elif good in goods_data.vanilla_df.index:
            price = float(goods_data.vanilla_df.loc[good, "default_market_price"])
        cost = amount * price
        good_costs.append((good, amount, cost))

    total_cost = sum(c for _, _, c in good_costs)
    for good, amount, cost in good_costs:
        pct_of_food = (cost / total_cost * 100) if total_cost > 0 else 0.0
        food_contributed = (pct_of_food / 100) * food_out
        food_per_unit = food_contributed / amount if amount > 0 else np.nan
        food_per_gold = food_contributed / cost if cost > 0 else np.nan
        rows_first.append({
            "PM": pm_name,
            "Input_Good": good,
            "Amount": amount,
            "Cost": cost,
            "Pct_of_food": pct_of_food,
            "Food_contributed": food_contributed,
            "Food_per_unit": food_per_unit,
            "Food_per_gold": food_per_gold,
        })

df_first = pd.DataFrame(rows_first)
# First DF: per-recipe, per-good: Amount, Cost, % of finished food (by cost share)
display(df_first.style.format({"Amount": "{:.3f}", "Cost": "{:.3f}", "Pct_of_food": "{:.1f}%", "Food_contributed": "{:.2f}", "Food_per_unit": "{:.2f}", "Food_per_gold": "{:.2f}"}))
# Pivot: recipes x goods with pct (compact view)
df_recipe_pivot = df_first.pivot_table(index="PM", columns="Input_Good", values="Pct_of_food", aggfunc="first").fillna(0)
display(df_recipe_pivot.style.format("{:.1f}%"))

# Second DF: per-good aggregates of food_per_unit and food_per_gold
df_agg = df_first[df_first["Food_per_unit"].notna() & df_first["Food_per_gold"].notna()]
df_second = df_agg.groupby("Input_Good").agg(
    food_per_unit_avg=("Food_per_unit", "mean"),
    food_per_unit_min=("Food_per_unit", "min"),
    food_per_unit_max=("Food_per_unit", "max"),
    food_per_gold_avg=("Food_per_gold", "mean"),
    food_per_gold_min=("Food_per_gold", "min"),
    food_per_gold_max=("Food_per_gold", "max"),
    n_recipes=("Food_per_unit", "count"),
).round(2)
display(df_second)

In [ ]:
# df_second.sort_values("food_per_unit_avg", ascending=False)
df_second.sort_values("food_per_gold_avg", ascending=False)

## Tavern

In [ ]:
tavern_analysis = analyze_building("tavern")
for df in tavern_analysis: display(df)

In [ ]:
# Tavern calculator: set parameters and run to get the same-style DF
victuals = 1.0
food_produced = 60.0
employment = 1.0
v_good = goods_data.get_good("victuals") if goods_data else None
victuals_price = float(v_good.get("default_market_price", 3.0)) if v_good is not None else 3.0
production_efficiency = 0.0

# Tavern output is abstract food; value at FOOD_PRICE (defines), not market price
food_good_name = "food"
food_price = float(defines_data.get_define("NMarket", "FOOD_PRICE", 0.05)) if defines_data else 0.05
pop_info = pop_data.get_pop_type("peasants") if pop_data else None
pop_food_consumption = float(pop_info.get("pop_food_consumption", 0.6)) if pop_info is not None else 0.6

worker_food_cost = employment * pop_food_consumption * food_price
victuals_cost = victuals * victuals_price
input_cost = victuals_cost + worker_food_cost
output_value = food_produced * (1 + production_efficiency) * food_price
profit = output_value - input_cost
margin = (output_value / input_cost) - 1 if input_cost > 0 else 0.0
epe = (input_cost / (food_produced * food_price)) - 1 if (food_produced * food_price) > 0 else 0.0

out_col = f"Out_{food_good_name}"
tavern_calc_df = pd.DataFrame([{
    "Version": "Calculator",
    "PM": "tavern_maintenance",
    "Pop_Type": "peasants",
    "Employment (1k)": employment,
    "EPE": epe,
    "Profit": profit,
    "Margin": margin,
    "Input_Cost": input_cost,
    "Output_Val": output_value,
    "Worker_Food_Cost": worker_food_cost,
    "output_price": food_price,
    "In_victuals": victuals,
    out_col: food_produced * (1 + production_efficiency)
}])

meta = ["Version", "PM", "Pop_Type", "Employment (1k)", "EPE"]
econ = ["Profit", "Margin", "Input_Cost", "Output_Val", "Worker_Food_Cost", "output_price"]
goods = ["In_victuals", out_col]
display(tavern_calc_df[meta + econ + goods].fillna(0))


In [ ]:
# I think the tavern should be +- 0 at food baseline price? This also means that all global food multipliers will benefit tavern and subsistence
